# Lesson 07 - Planning Design Pattern

Denna notebook visar **Planning Design Pattern** för AI-agenter med Microsoft Agent Framework.  
Du kommer att lära dig hur man delar upp komplexa reseförfrågningar i strukturerade deluppgifter, tilldelar dem till specialistagenter,  
och utför den resulterande planen — allt med strukturerad output som drivs av Pydantic-modeller.


## Installation


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Uppgiftsuppdelning

Uppgiftsuppdelning är kärnan i planeringsdesignmönstret. Istället för att be en enda agent att hantera en komplex förfrågan från början till slut, delar vi upp problemet i mindre, väl definierade **underuppgifter**. Varje underuppgift tilldelas en specialistagent (t.ex. flyg, hotell, aktiviteter) med tydliga prioriteringar och beroendeordning.

Detta tillvägagångssätt ger flera fördelar:
- **Tydlighet**: varje underuppgift har ett enda ansvar.
- **Parallellism**: oberoende underuppgifter kan köras samtidigt.
- **Tillförlitlighet**: fel isoleras till enskilda underuppgifter.
- **Budgetuppföljning**: kostnader uppskattas per underuppgift och summeras.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Skapa en planeringsagent med strukturerad output

Planeringsagenten fungerar som en **receptionskoordinator**. Givet en övergripande reseförfrågan producerar den en strukturerad `TravelPlan` — som delar upp förfrågan i deluppgifter, sätter prioriteringar och identifierar beroenden så att en concierge eller en exekveringsnivå kan utföra arbetet.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Utföra en plan med specialistverktyg

När receptionisten har tagit fram en strukturerad plan, utför **concierge-agenten** den.  
Varje specialistverktyg hanterar en kategori av deluppgifter (flyg, hotell, aktiviteter). Concierge-agenten går igenom planens deluppgifter i beroendeordning och skickar varje uppgift till rätt verktyg.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Summary

I den här lektionen lärde du dig **Planeringsdesignmönstret** för AI-agenter:

1. **Uppgiftsuppdelning** — En receptionsplaneringsagent delar upp en komplex reseförfrågan i
   strukturerade deluppgifter med hjälp av Pydantic-modeller och tilldelar varje uppgift till en specialistagent med prioriteringar
   och beroenden.
2. **Strukturerat resultat** — Genom att skicka ett `response_format` returnerar agenten ett validerat
   `TravelPlan`-objekt istället för fri text, vilket gör efterföljande bearbetning pålitlig.
3. **Planutförande** — En concierge-agent går igenom deluppgifterna med hjälp av specialistverktyg
   (`book_flight`, `reserve_hotel`, `book_activity`) för att genomföra planen och rapportera resultat.

Detta mönster separerar *vad som ska göras* (planering) från *hur det ska göras* (utförande), vilket gör agenter
mer modulära, testbara och enklare att utvidga.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, bör du vara medveten om att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För viktig information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för eventuella missförstånd eller feltolkningar som uppstår genom användning av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
